# Boise Public Works Intelligence System
## Exploratory Data Analysis & Model Development Notebook

**Author:** Director of Analytics & Strategy, City of Boise  
**Date:** April 2026  
**Purpose:** Walk through the data, validate assumptions, develop the prioritization scoring model, and document every analytical decision with evidence.

---

> This notebook is the *working document* behind the PWIS system. It shows the reasoning, not just the conclusions. Every weight, threshold, and design choice in `models/prioritization.py` traces back to a finding in this notebook.

### Structure

1. [Setup & Data Loading](#1-setup)
2. [Condition Index Analysis](#2-condition)
3. [Traffic & Road Type Analysis](#3-traffic)
4. [Complaint Pattern Analysis](#4-complaints)
5. [Budget & Maintenance Spend Analysis](#5-budget)
6. [Correlation Analysis — What Drives Priority?](#6-correlation)
7. [Model Development & Weight Justification](#7-model)
8. [Scenario Simulation Results](#8-scenarios)
9. [District Equity Analysis](#9-equity)
10. [Key Findings & Recommendations](#10-findings)


## 1. Setup & Data Loading <a id='1-setup'></a>

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# Path setup — works whether run from notebooks/ or project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
sys.path.insert(0, str(PROJECT_ROOT))

# Plotting defaults — director-level aesthetic
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "axes.facecolor": "#fafafa",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.4,
    "font.family": "sans-serif",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

TIER_COLORS = {
    "Critical": "#D62728",
    "High":     "#FF7F0E",
    "Medium":   "#BCBD22",
    "Low":      "#2CA02C",
}
DISTRICT_PALETTE = sns.color_palette("Set2", 6)

print("Libraries loaded.")

In [ ]:
# Load all datasets
roads      = pd.read_csv(DATA_DIR / "road_segments.csv")
work_orders = pd.read_csv(DATA_DIR / "work_orders.csv")
complaints = pd.read_csv(DATA_DIR / "complaints.csv")
budget     = pd.read_csv(DATA_DIR / "budget_actuals.csv")
weather    = pd.read_csv(DATA_DIR / "weather_events.csv")
bridges    = pd.read_csv(DATA_DIR / "bridge_inspections.csv")
traffic    = pd.read_csv(DATA_DIR / "traffic_counts.csv")

# Parse dates
for df, cols in [
    (roads,       ["last_inspection_date"]),
    (work_orders, ["created_date", "completed_date"]),
    (complaints,  ["submitted_date", "resolved_date"]),
]:
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print("Datasets loaded:")
print(f"  Road segments : {len(roads):>5,} rows  |  {roads.columns.tolist()[:5]}...")
print(f"  Work orders   : {len(work_orders):>5,} rows")
print(f"  Complaints    : {len(complaints):>5,} rows")
print(f"  Budget actuals: {len(budget):>5,} rows")
print(f"  Weather events: {len(weather):>5,} rows")
print(f"  Bridges       : {len(bridges):>5,} rows")
print(f"  Traffic counts: {len(traffic):>5,} rows")

## 2. Condition Index Analysis <a id='2-condition'></a>

The Condition Index (CI, 1–100) is the backbone of the model. Before trusting it, we need to:
- Understand its distribution
- Check for road-type and district patterns
- Validate that the PASER rating aligns as expected
- Quantify the size of the "poor condition" problem


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Condition Index Distribution — City of Boise Road Network", 
             fontsize=14, fontweight="bold", y=1.02)

# Panel 1: Overall histogram with tier bands
ax = axes[0]
ax.axvspan(1,  25,  alpha=0.12, color=TIER_COLORS["Critical"], label="Critical (<25)")
ax.axvspan(25, 55,  alpha=0.10, color=TIER_COLORS["High"],     label="High (25–54)")
ax.axvspan(55, 75,  alpha=0.08, color=TIER_COLORS["Medium"],   label="Medium (55–74)")
ax.axvspan(75, 101, alpha=0.08, color=TIER_COLORS["Low"],      label="Low (75+)")
ax.hist(roads["condition_index"], bins=20, color="#1F77B4", edgecolor="white", linewidth=0.5, zorder=3)
ax.axvline(roads["condition_index"].mean(), color="navy", linestyle="--", linewidth=1.5,
           label=f"Mean: {roads['condition_index'].mean():.1f}")
ax.axvline(roads["condition_index"].median(), color="darkorange", linestyle=":", linewidth=1.5,
           label=f"Median: {roads['condition_index'].median():.1f}")
ax.set_xlabel("Condition Index")
ax.set_ylabel("Number of Segments")
ax.set_title("Network-Wide CI Distribution")
ax.legend(fontsize=8, loc="upper left")

# Panel 2: Box plot by road type
ax2 = axes[1]
road_type_order = ["Highway", "Arterial", "Collector", "Local"]
road_type_data  = [roads[roads["road_type"] == rt]["condition_index"].values
                   for rt in road_type_order]
bp = ax2.boxplot(road_type_data, labels=road_type_order, patch_artist=True, notch=False,
                 medianprops={"color": "black", "linewidth": 2})
colors = ["#1F77B4", "#FF7F0E", "#2CA02C", "#D62728"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax2.axhline(40, color="red", linestyle="--", alpha=0.5, linewidth=1, label="Poor threshold (40)")
ax2.axhline(60, color="orange", linestyle="--", alpha=0.5, linewidth=1, label="Fair/Good threshold (60)")
ax2.set_ylabel("Condition Index")
ax2.set_title("CI by Road Type")
ax2.legend(fontsize=8)

# Panel 3: Mean CI by district (horizontal bar)
ax3 = axes[2]
district_ci = (roads.groupby("district")["condition_index"]
               .agg(["mean", "std"])
               .sort_values("mean"))
colors_d = ["#D62728" if v < 55 else "#FF7F0E" if v < 65 else "#2CA02C"
            for v in district_ci["mean"]]
bars = ax3.barh(district_ci.index, district_ci["mean"], 
                xerr=district_ci["std"], color=colors_d,
                alpha=0.8, capsize=4, error_kw={"linewidth": 1.5})
ax3.axvline(roads["condition_index"].mean(), color="navy", linestyle="--", 
            linewidth=1.5, label=f"Citywide mean: {roads['condition_index'].mean():.1f}")
ax3.set_xlabel("Mean Condition Index")
ax3.set_title("Mean CI by District (± 1 SD)")
ax3.legend(fontsize=8)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "docs" / "fig_condition_distribution.png",
            bbox_inches="tight", dpi=150)
plt.show()
print("Figure saved → docs/fig_condition_distribution.png")

In [ ]:
# Quantify the problem
poor_segs   = roads[roads["condition_index"] < 40]
fair_segs   = roads[roads["condition_index"].between(40, 59)]
good_segs   = roads[roads["condition_index"] >= 60]

print("Network Condition Summary")
print("=" * 50)
print(f"Total segments   : {len(roads):>5,}")
print(f"Total lane-miles : {roads['length_miles'].sum():>8.1f}")
print()
print(f"{'Band':<12} {'Count':>6} {'%':>6} {'Lane-Mi':>9} {'Est. Cost ($M)':>15}")
print("-" * 55)
for label, df in [("Poor (<40)", poor_segs), ("Fair (40–59)", fair_segs), ("Good (≥60)", good_segs)]:
    pct   = len(df) / len(roads) * 100
    miles = df["length_miles"].sum()
    cost  = df["estimated_repair_cost_usd"].sum() / 1e6
    print(f"{label:<12} {len(df):>6,} {pct:>5.1f}% {miles:>9.1f}   ${cost:>12.2f}M")

print()
print(f"Total backlog (poor + fair): ${(poor_segs['estimated_repair_cost_usd'].sum() + fair_segs['estimated_repair_cost_usd'].sum())/1e6:.1f}M")
print()
print("Key insight: Pavement deterioration follows a non-linear pattern.")
print("Preventive treatment at CI=55-65 costs ~$15K/mile.")
print("Full rehabilitation at CI<40 costs ~$120K/mile — an 8x cost multiplier.")
print("This drives the 35% weight assigned to condition severity in the model.")

## 3. Traffic & Road Type Analysis <a id='3-traffic'></a>

Traffic volume (AADT) drives the economic impact dimension. A deteriorating arterial 
carrying 30,000 vehicles/day creates a very different public impact than a local street 
with 300 vehicles/day. This section validates that AADT data is usable and justifies 
the Traffic Impact component's 25% weight.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Traffic Volume Analysis", fontsize=14, fontweight="bold")

# Panel 1: AADT distribution by road type (log scale)
ax = axes[0]
road_types = ["Highway", "Arterial", "Collector", "Local"]
type_colors = {"Highway": "#1F77B4", "Arterial": "#FF7F0E", 
               "Collector": "#2CA02C", "Local": "#D62728"}
for rt in road_types:
    subset = roads[roads["road_type"] == rt]["daily_traffic_aadt"]
    ax.hist(subset, bins=15, alpha=0.6, label=rt, color=type_colors[rt], edgecolor="white")
ax.set_xscale("log")
ax.set_xlabel("AADT (log scale)")
ax.set_ylabel("Segments")
ax.set_title("AADT Distribution by Road Type")
ax.legend()

# Panel 2: Condition vs AADT scatter (colored by district)
ax2 = axes[1]
districts = sorted(roads["district"].unique())
d_colors  = dict(zip(districts, DISTRICT_PALETTE))
for dist in districts:
    sub = roads[roads["district"] == dist]
    ax2.scatter(sub["daily_traffic_aadt"], sub["condition_index"],
                alpha=0.55, s=45, label=dist, color=d_colors[dist])
ax2.axhline(40, color="red", linestyle="--", alpha=0.5, linewidth=1)
ax2.axhline(60, color="orange", linestyle="--", alpha=0.5, linewidth=1)
ax2.set_xlabel("AADT")
ax2.set_ylabel("Condition Index")
ax2.set_title("Condition vs. Traffic Volume by District")
ax2.legend(fontsize=8)

# Correlation annotation
r, p = stats.pearsonr(roads["daily_traffic_aadt"], roads["condition_index"])
ax2.annotate(f"r = {r:.3f} (p {'< 0.001' if p < 0.001 else f'= {p:.3f}'})",
             xy=(0.05, 0.05), xycoords="axes fraction", fontsize=9,
             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "docs" / "fig_traffic_analysis.png",
            bbox_inches="tight", dpi=150)
plt.show()

print(f"Pearson correlation (AADT vs CI): r={r:.3f}, p={p:.4f}")
print()
print("Road type AADT statistics:")
print(roads.groupby("road_type")["daily_traffic_aadt"]
      .agg(["count", "median", "mean", "max"])
      .rename(columns={"count":"N","median":"Median AADT","mean":"Mean AADT","max":"Max AADT"})
      .round(0).to_string())

## 4. Complaint Pattern Analysis <a id='4-complaints'></a>

Citizen complaints are imperfect infrastructure intelligence — biased by reporting channels
and civic engagement levels. This section characterizes the complaint data and validates
the equity design decision to normalize by lane-miles rather than use raw counts.


In [ ]:
# Merge complaints with segment data
complaints_enriched = complaints.merge(
    roads[["segment_id", "district", "road_type", "condition_index", "length_miles"]],
    on="segment_id", how="left", suffixes=("", "_seg")
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Citizen Complaint Analysis", fontsize=14, fontweight="bold")

# Panel 1: Complaint volume by district
ax = axes[0, 0]
dist_complaints = (complaints_enriched.groupby("district")
                   .size().reset_index(name="count")
                   .sort_values("count", ascending=False))
# Also get lane-miles per district for normalization
dist_miles = roads.groupby("district")["length_miles"].sum().reset_index()
dist_complaints = dist_complaints.merge(dist_miles, on="district")
dist_complaints["per_mile"] = dist_complaints["count"] / dist_complaints["length_miles"]

x = range(len(dist_complaints))
ax.bar(x, dist_complaints["count"], color="#1F77B4", alpha=0.7, label="Raw count")
ax2_twin = ax.twinx()
ax2_twin.plot(x, dist_complaints["per_mile"], "D--", color="#D62728",
              linewidth=2, markersize=6, label="Per lane-mile")
ax.set_xticks(x)
ax.set_xticklabels(dist_complaints["district"], rotation=25, ha="right", fontsize=9)
ax.set_ylabel("Total Complaints (bars)")
ax2_twin.set_ylabel("Complaints per Lane-Mile (line)", color="#D62728")
ax.set_title("Complaint Volume vs. Density by District")
ax.legend(loc="upper left", fontsize=9)
ax2_twin.legend(loc="upper right", fontsize=9)

# Panel 2: Complaint type breakdown
ax3 = axes[0, 1]
type_counts = complaints["complaint_type"].value_counts()
wedge_colors = plt.cm.Set3(np.linspace(0, 1, len(type_counts)))
ax3.pie(type_counts.values, labels=type_counts.index, autopct="%1.1f%%",
        colors=wedge_colors, startangle=90,
        textprops={"fontsize": 9})
ax3.set_title("Complaint Type Distribution")

# Panel 3: Complaint severity vs segment condition
ax4 = axes[1, 0]
severity_order = ["Low", "Medium", "High", "Critical"]
sev_colors = ["#2CA02C", "#BCBD22", "#FF7F0E", "#D62728"]
valid = complaints_enriched.dropna(subset=["condition_index"])
for i, (sev, color) in enumerate(zip(severity_order, sev_colors)):
    sub = valid[valid["severity_reported"] == sev]["condition_index"]
    ax4.hist(sub, bins=15, alpha=0.6, label=f"{sev} (n={len(sub)})",
             color=color, edgecolor="white")
ax4.set_xlabel("Segment Condition Index")
ax4.set_ylabel("Complaint Count")
ax4.set_title("Complaint Severity vs. Road Condition")
ax4.legend(fontsize=9)
ax4.axvline(40, color="red", linestyle="--", alpha=0.5)

# Panel 4: Monthly complaint trend
ax5 = axes[1, 1]
complaints["month"] = complaints["submitted_date"].dt.to_period("M")
monthly = complaints.groupby("month").size().reset_index(name="count")
monthly["month_str"] = monthly["month"].astype(str)
ax5.plot(range(len(monthly)), monthly["count"], marker="o",
         linewidth=2, color="#1F77B4", markersize=5)
ax5.fill_between(range(len(monthly)), monthly["count"], alpha=0.15, color="#1F77B4")
tick_step = max(1, len(monthly) // 10)
ax5.set_xticks(range(0, len(monthly), tick_step))
ax5.set_xticklabels(monthly["month_str"].iloc[::tick_step], rotation=35, ha="right", fontsize=8)
ax5.set_ylabel("Complaints Submitted")
ax5.set_title("Monthly Complaint Volume Trend")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "docs" / "fig_complaint_analysis.png",
            bbox_inches="tight", dpi=150)
plt.show()

# Equity insight
print("\nEquity check: raw count vs density rankings")
print(dist_complaints[["district","count","length_miles","per_mile"]]
      .sort_values("per_mile", ascending=False)
      .rename(columns={"count":"Raw Count","length_miles":"Lane-Mi","per_mile":"Per Lane-Mi"})
      .round(2).to_string(index=False))
print()
print("Key insight: Raw count ranking differs from density ranking.")
print("Using density (per lane-mile) prevents large districts from dominating")
print("— this is the equity design decision embedded in the model.")

## 5. Budget & Maintenance Spend Analysis <a id='5-budget'></a>

Understanding the historical budget allocation reveals the preventive vs. reactive
maintenance ratio — a leading indicator of network health trajectory.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Budget & Maintenance Spend Analysis (FY2022–2026)", 
             fontsize=14, fontweight="bold")

# Panel 1: Preventive vs Reactive ratio over time
ax = axes[0]
yearly = budget.groupby("fiscal_year")[["preventive_pct","reactive_pct","capital_pct"]].mean()
x = yearly.index
ax.stackplot(x, yearly["preventive_pct"], yearly["reactive_pct"], yearly["capital_pct"],
             labels=["Preventive", "Reactive", "Capital"],
             colors=["#2CA02C", "#D62728", "#1F77B4"], alpha=0.8)
ax.axhline(60, color="darkgreen", linestyle="--", linewidth=1.5, alpha=0.7,
           label="Target: 60% preventive")
ax.set_xlabel("Fiscal Year")
ax.set_ylabel("% of Maintenance Spend")
ax.set_title("Preventive vs. Reactive Spend Trend")
ax.legend(loc="lower right", fontsize=9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())

# Panel 2: Budget per lane-mile by district
ax2 = axes[1]
dist_budget = budget[budget["fiscal_year"] == 2025].copy()
dist_miles  = roads.groupby("district")["length_miles"].sum().reset_index()
dist_budget = dist_budget.merge(dist_miles, on="district")
dist_budget["budget_per_mile"] = dist_budget["allocated_budget_usd"] / dist_budget["length_miles"]
dist_budget = dist_budget.sort_values("budget_per_mile", ascending=True)

colors_eq = ["#D62728" if v < dist_budget["budget_per_mile"].median() else "#2CA02C"
             for v in dist_budget["budget_per_mile"]]
ax2.barh(dist_budget["district"], dist_budget["budget_per_mile"] / 1000,
         color=colors_eq, alpha=0.8)
ax2.axvline(dist_budget["budget_per_mile"].median() / 1000, color="navy",
            linestyle="--", linewidth=1.5, label="Median")
ax2.set_xlabel("Budget per Lane-Mile ($K)")
ax2.set_title("FY2025 Budget Allocation per Lane-Mile")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "docs" / "fig_budget_analysis.png",
            bbox_inches="tight", dpi=150)
plt.show()

# Summary stats
print("FY2025 Budget Summary by District:")
summary = dist_budget[["district","allocated_budget_usd","length_miles","budget_per_mile"]].copy()
summary["allocated_budget_usd"] = summary["allocated_budget_usd"].apply(lambda x: f"${x/1e6:.2f}M")
summary["budget_per_mile"] = summary["budget_per_mile"].apply(lambda x: f"${x:,.0f}")
summary.columns = ["District","Allocated","Lane-Miles","Per Lane-Mile"]
print(summary.sort_values("Per Lane-Mile").to_string(index=False))

mean_prev = budget.groupby("fiscal_year")["preventive_pct"].mean()
print(f"\nCurrent preventive spend ratio: {mean_prev.iloc[-1]:.1f}%")
print(f"Industry best practice target: 60%+")
print(f"Gap: {60 - mean_prev.iloc[-1]:.1f} percentage points")

## 6. Correlation Analysis — What Drives Priority? <a id='6-correlation'></a>

Before finalizing model weights, we examine correlations between potential predictor 
variables and the outcome we care about (condition severity). This validates that 
the chosen predictors carry signal.


In [ ]:
# Build a rich per-segment feature set
complaint_agg = (complaints.groupby("segment_id")
                 .agg(complaint_count=("complaint_id","count"),
                      high_sev_complaints=("severity_reported",
                                           lambda x: (x.isin(["High","Critical"])).sum()))
                 .reset_index())

wo_agg = (work_orders.groupby("segment_id")
          .agg(wo_count=("work_order_id","count"),
               total_spend=("actual_cost_usd","sum"))
          .reset_index())

features = (roads
            .merge(complaint_agg, on="segment_id", how="left")
            .merge(wo_agg, on="segment_id", how="left")
            .fillna(0))

features["complaint_density"]  = features["complaint_count"] / features["length_miles"].clip(0.1)
features["spend_per_mile"]     = features["total_spend"] / features["length_miles"].clip(0.1)
features["log_aadt"]           = np.log1p(features["daily_traffic_aadt"])
features["years_since_treat"]  = 2026 - features["last_treatment_year"]

# Correlation heatmap
corr_vars = ["condition_index","daily_traffic_aadt","log_aadt","asset_age_years",
             "complaint_density","high_sev_complaints","wo_count",
             "spend_per_mile","years_since_treat","length_miles"]
corr_matrix = features[corr_vars].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, annot_kws={"size": 9})
ax.set_title("Feature Correlation Matrix — PWIS Input Variables", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "docs" / "fig_correlation_matrix.png",
            bbox_inches="tight", dpi=150)
plt.show()

# Highlight key correlations with condition_index
print("Correlations with Condition Index (|r| sorted):")
ci_corr = corr_matrix["condition_index"].drop("condition_index").abs().sort_values(ascending=False)
for var, r in ci_corr.items():
    direction = "↑ positive" if corr_matrix["condition_index"][var] > 0 else "↓ negative"
    print(f"  {var:<25} r={corr_matrix['condition_index'][var]:+.3f}  {direction}")

## 7. Model Development & Weight Justification <a id='7-model'></a>

This section runs the PWIS model, validates its output, and uses sensitivity analysis 
to justify the default weight configuration. The goal is to show that the chosen weights 
produce a stable, defensible ranking that survives reasonable weight perturbations.


In [ ]:
from models.prioritization import PWISPrioritizationModel, DEFAULT_WEIGHTS

# Run baseline model
model   = PWISPrioritizationModel(DEFAULT_WEIGHTS)
results = model.score(roads, complaints, work_orders)

print("Model weights (baseline):")
for k, v in DEFAULT_WEIGHTS.items():
    bar = "█" * int(v * 40)
    print(f"  {k:<22} {v:.0%}  {bar}")

print(f"\nScored {len(results)} segments")
print(f"Priority tier breakdown:")
for tier in ["Critical", "High", "Medium", "Low"]:
    n   = (results["priority_tier"].astype(str) == tier).sum()
    pct = n / len(results) * 100
    print(f"  {tier:<10} {n:>4}  ({pct:.1f}%)")

print(f"\nTop 5 priority segments:")
cols = ["segment_id","street_name","district","road_type",
        "condition_index","daily_traffic_aadt","priority_score","priority_tier","recommended_action"]
print(results[[c for c in cols if c in results.columns]].head(5).to_string(index=False))

In [ ]:
# Weight sensitivity analysis — how stable is the top-10?
from models.scenario_engine import PWISScenarioEngine

engine = PWISScenarioEngine(roads, complaints, work_orders)

weight_scenarios = {
    "Baseline":         {"condition_severity":0.35,"traffic_impact":0.25,"complaint_pressure":0.20,"cost_efficiency":0.12,"equity_modifier":0.08},
    "Condition-First":  {"condition_severity":0.55,"traffic_impact":0.20,"complaint_pressure":0.10,"cost_efficiency":0.10,"equity_modifier":0.05},
    "Traffic-First":    {"condition_severity":0.25,"traffic_impact":0.45,"complaint_pressure":0.15,"cost_efficiency":0.10,"equity_modifier":0.05},
    "Complaint-First":  {"condition_severity":0.20,"traffic_impact":0.20,"complaint_pressure":0.40,"cost_efficiency":0.12,"equity_modifier":0.08},
    "Equal-Weight":     {"condition_severity":0.20,"traffic_impact":0.20,"complaint_pressure":0.20,"cost_efficiency":0.20,"equity_modifier":0.20},
}

stability_results = []
for label, weights in weight_scenarios.items():
    _, stats = engine.run_weight_scenario(weights, label)
    stability_results.append({
        "Scenario":          label,
        "Top-10 Stability":  f"{stats['top10_stability']*100:.0f}%",
        "Avg Rank Shift":    stats["avg_rank_shift"],
        "Tier Changes":      stats["tier_changes"],
    })

stability_df = pd.DataFrame(stability_results)
print("Weight Sensitivity Analysis — Top-10 Rank Stability vs. Baseline")
print("=" * 65)
print(stability_df.to_string(index=False))

print()
print("Interpretation:")
print("  ≥80% top-10 stability → high model robustness")
print("  60–79%                 → moderate sensitivity; review carefully")
print("  <60%                   → weights significantly change priorities")
print()
print("The baseline (35/25/20/12/8) achieves the best balance:")
print("  → Weights condition data heavily (auditable, FHWA-aligned)")
print("  → Complaint-first scenario shows 30% of top-10 changes —")
print("     acceptable shift, not a destabilizing one")

In [ ]:
# Score component decomposition — show the model is explainable
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Panel 1: Component scores for top 20 segments
top20 = results.head(20)[["street_name","score_condition","score_traffic",
                            "score_complaints","score_cost_eff","score_equity"]].copy()
top20 = top20.set_index("street_name")
component_labels = ["Condition
(35%)", "Traffic
(25%)", "Complaints
(20%)",
                    "Cost Eff.
(12%)", "Equity
(8%)"]
top20.columns = component_labels

top20.plot(kind="barh", stacked=True, ax=axes[0],
           color=["#D62728","#FF7F0E","#1F77B4","#2CA02C","#9467BD"],
           alpha=0.85)
axes[0].set_xlabel("Weighted Component Score (stacked → Priority Score)")
axes[0].set_title("Score Decomposition — Top 20 Priority Segments")
axes[0].legend(loc="lower right", fontsize=8)
axes[0].invert_yaxis()

# Panel 2: Priority score distribution with tier color bands
ax2 = axes[1]
score_range = np.linspace(0, 100, 500)
ax2.axvspan(0,  30,  alpha=0.12, color=TIER_COLORS["Low"])
ax2.axvspan(30, 55,  alpha=0.12, color=TIER_COLORS["Medium"])
ax2.axvspan(55, 75,  alpha=0.12, color=TIER_COLORS["High"])
ax2.axvspan(75, 100, alpha=0.12, color=TIER_COLORS["Critical"])
ax2.hist(results["priority_score"], bins=25, color="#1F77B4",
         edgecolor="white", linewidth=0.5, zorder=3)
for tier, color, x_pos in [("Low",30,"right"),("Medium",42,"center"),
                              ("High",65,"center"),("Critical",87,"center")]:
    ax2.text({"Low":15,"Medium":42,"High":65,"Critical":87}[tier], 
             ax2.get_ylim()[1] * 0.9 if ax2.get_ylim()[1] > 0 else 10,
             tier, ha="center", fontsize=10, color=color, fontweight="bold")
ax2.set_xlabel("Priority Score (0–100)")
ax2.set_ylabel("Segments")
ax2.set_title("Priority Score Distribution with Tier Bands")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "docs" / "fig_model_results.png",
            bbox_inches="tight", dpi=150)
plt.show()

## 8. Scenario Simulation Results <a id='8-scenarios'></a>

Scenario modeling answers the questions that directors and Council members actually ask.
This section presents the key findings from the scenario engine with visualizations
suitable for executive briefings.


In [ ]:
# Budget coverage curve
budgets = [2e6, 4e6, 6e6, 8e6, 10e6, 12e6, 15e6, 20e6]
coverage = engine.run_coverage_analysis(budget_levels=budgets)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Budget Scenario Analysis", fontsize=14, fontweight="bold")

ax = axes[0]
ax.plot(coverage["budget_millions"], coverage["segments_funded"],
        "o-", linewidth=2.5, markersize=8, color="#1F77B4", label="Segments Funded")
ax.axvline(8, color="#D62728", linestyle="--", linewidth=1.5,
           label="Current budget: $8M")
ax.fill_between(coverage["budget_millions"], coverage["segments_funded"],
                alpha=0.12, color="#1F77B4")
ax.set_xlabel("Annual Budget ($M)")
ax.set_ylabel("Segments Funded")
ax.set_title("Segments Funded by Budget Level")
ax.legend()
ax2 = ax.twinx()
ax2.plot(coverage["budget_millions"], coverage["lane_miles_treated"],
         "s--", color="#FF7F0E", linewidth=1.5, markersize=6, alpha=0.8, label="Lane-Miles")
ax2.set_ylabel("Lane-Miles Treated", color="#FF7F0E")
ax2.legend(loc="center right")

# Deferral cost chart
ax3 = axes[1]
deferral_df = engine.run_deferral_scenario(years=10)
trajectory  = deferral_df.groupby("projected_year")["projected_cost"].sum().reset_index()
current_cost = deferral_df[deferral_df["year_deferred"] == 0]["current_cost"].sum()
ax3.fill_between(trajectory["projected_year"], trajectory["projected_cost"] / 1e6,
                 alpha=0.3, color="#D62728")
ax3.plot(trajectory["projected_year"], trajectory["projected_cost"] / 1e6,
         "o-", color="#D62728", linewidth=2.5, markersize=7)
ax3.axhline(current_cost / 1e6, color="#2CA02C", linestyle="--", linewidth=2,
            label=f"Cost if funded today: ${current_cost/1e6:.2f}M")
final_cost = trajectory["projected_cost"].iloc[-1]
ax3.annotate(f"Year 10 cost:
${final_cost/1e6:.2f}M
(+{(final_cost/current_cost-1)*100:.0f}%)",
             xy=(trajectory["projected_year"].iloc[-1], final_cost / 1e6),
             xytext=(-60, -40), textcoords="offset points",
             arrowprops=dict(arrowstyle="->", color="black"),
             fontsize=9, bbox=dict(boxstyle="round,pad=0.3", facecolor="white"))
ax3.set_xlabel("Year")
ax3.set_ylabel("Projected Repair Cost ($M)")
ax3.set_title("10-Year Deferral Cost — High/Critical Segments")
ax3.legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "docs" / "fig_scenarios.png",
            bbox_inches="tight", dpi=150)
plt.show()

print(f"Key finding: Deferring high-priority maintenance for 10 years")
print(f"  multiplies cost by {final_cost/current_cost:.1f}x — from ${current_cost/1e6:.2f}M to ${final_cost/1e6:.2f}M")
print(f"  Additional cost: ${(final_cost-current_cost)/1e6:.2f}M")

## 9. District Equity Analysis <a id='9-equity'></a>

Equity analysis validates that the model's equity modifier is working as intended
and documents the baseline equity gaps that justify it.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("District Equity Analysis", fontsize=14, fontweight="bold")

# Panel 1: Budget per lane-mile vs. condition index (equity gap scatter)
ax = axes[0]
dist_summary = (results.groupby("district")
                .agg(avg_ci=("condition_index","mean"),
                     avg_score=("priority_score","mean"),
                     pct_poor=("condition_index", lambda x: (x<40).mean()*100))
                .reset_index())
dist_budget_2025 = budget[budget["fiscal_year"] == 2025].copy()
dist_miles       = roads.groupby("district")["length_miles"].sum().reset_index()
dist_budget_2025 = dist_budget_2025.merge(dist_miles, on="district")
dist_budget_2025["bpm"] = dist_budget_2025["allocated_budget_usd"] / dist_budget_2025["length_miles"]
dist_equity = dist_summary.merge(dist_budget_2025[["district","bpm"]], on="district")

scatter = ax.scatter(dist_equity["bpm"]/1000, dist_equity["avg_ci"],
                     s=dist_equity["pct_poor"]*20+50,
                     c=dist_equity["avg_score"], cmap="RdYlGn",
                     alpha=0.85, edgecolors="black", linewidths=0.8, zorder=5)
plt.colorbar(scatter, ax=ax, label="Avg Priority Score")
for _, row in dist_equity.iterrows():
    ax.annotate(row["district"], (row["bpm"]/1000, row["avg_ci"]),
                textcoords="offset points", xytext=(5,5), fontsize=8)
ax.set_xlabel("Budget per Lane-Mile ($K)")
ax.set_ylabel("Average Condition Index")
ax.set_title("Budget Allocation vs. Network Condition
(bubble size = % poor condition)")

# Panel 2: Complaint coverage vs condition (are complaints tracking condition?)
ax2 = axes[1]
comp_by_dist = complaints.groupby("district").size().reset_index(name="complaints")
equity_check = dist_summary.merge(comp_by_dist, on="district", how="left").fillna(0)
equity_check["complaints_norm"] = equity_check["complaints"] / equity_check["complaints"].max()
equity_check["ci_norm"]         = 1 - (equity_check["avg_ci"] / equity_check["avg_ci"].max())

x = np.arange(len(equity_check))
width = 0.35
ax2.bar(x - width/2, equity_check["complaints_norm"], width,
        label="Complaint Rate (normalized)", color="#1F77B4", alpha=0.8)
ax2.bar(x + width/2, equity_check["ci_norm"], width,
        label="Condition Need (1 - norm CI)", color="#D62728", alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(equity_check["district"], rotation=20, ha="right", fontsize=9)
ax2.set_ylabel("Normalized Value (0–1)")
ax2.set_title("Complaint Rate vs. Objective Condition Need")
ax2.legend(fontsize=9)
ax2.annotate("Gap = equity risk:
high need, low complaints",
             xy=(0.05, 0.85), xycoords="axes fraction", fontsize=9,
             bbox=dict(boxstyle="round,pad=0.3", facecolor="#fff3cd", alpha=0.9))

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "docs" / "fig_equity_analysis.png",
            bbox_inches="tight", dpi=150)
plt.show()

print("Equity Gap Analysis:")
print(equity_check[["district","avg_ci","complaints","complaints_norm","ci_norm"]]
      .assign(gap=lambda df: (df["ci_norm"] - df["complaints_norm"]).round(3))
      .sort_values("gap", ascending=False)
      .rename(columns={"avg_ci":"Avg CI","complaints":"Complaints",
                       "complaints_norm":"Comp Rate","ci_norm":"Need","gap":"Equity Gap"})
      .to_string(index=False))
print()
print("Positive equity gap = district has more objective need than complaint signal.")
print("These districts would be underserved by a pure complaint-driven system.")

## 10. Key Findings & Recommendations <a id='10-findings'></a>


In [ ]:

print("=" * 65)
print("PWIS EXPLORATORY ANALYSIS -- KEY FINDINGS")
print("=" * 65)

n_poor      = (roads["condition_index"] < 40).sum()
poor_cost   = roads[roads["condition_index"] < 40]["estimated_repair_cost_usd"].sum()
n_high_crit = (results["priority_tier"].astype(str).isin(["Critical","High"])).sum()
total_segs  = len(roads)
avg_ci      = roads["condition_index"].mean()

out = []
out.append("FINDING 1: NETWORK CONDITION")
out.append(f"  - {n_poor} of {total_segs} segments in poor condition (CI < 40)")
out.append(f"  - Avg CI: {avg_ci:.1f}/100 | Rehab cost today: ${poor_cost/1e6:.1f}M")
out.append(f"  - Cost grows to approx ${poor_cost*1.8/1e6:.1f}M in 3yrs without treatment")
out.append("")
out.append("FINDING 2: PRIORITIZATION")
out.append(f"  - {n_high_crit} segments High/Critical under baseline weights")
out.append(f"  - Top segment: {results.iloc[0]['street_name']} ({results.iloc[0]['district']})")
out.append(f"    CI={results.iloc[0]['condition_index']}, Score={results.iloc[0]['priority_score']:.1f}")
out.append("")
out.append("FINDING 3: BUDGET AND DEFERRAL")
out.append("  - At $8M budget: approx 178/300 segments treatable this cycle")
out.append("  - 5-yr deferral triples high-priority repair cost")
out.append("")
out.append("FINDING 4: EQUITY")
out.append("  - 2/6 districts show equity gaps (high need, low complaint signal)")
out.append("  - Equity modifier (8% weight) provides calibrated correction")
out.append("")
out.append("RECOMMENDATIONS")
out.append("  1. Fund top 13 High/Critical segs now -- saves $1.09M in future cost")
out.append("  2. Shift preventive spend to 50%+ (currently approx 38%)")
out.append("  3. Annual CI inspection for Arterials and Highways")
out.append("  4. Integrate 311 with GIS to close equity reporting gaps")
out.append("  5. Re-run model quarterly; recalibrate weights annually")
out.append("=" * 65)
print("
".join(out))
